In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts', 'Dates']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives', 'JioSaavn']


In [3]:
from lib import allmusic
mio   = allmusic.MusicDBIO(verbose=False)
webio = allmusic.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant AllMusic DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AllMusic


In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
knownArtists       = mio.data.getSummaryNameData()
searchArtists      = mio.data.getSearchArtistData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Local Artists:             {0}".format(len(localArtists.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

AllMusic Search Results
   Global Master Search:      761013
   Global Master Search Data: 0
   Local Artists:             1544795
   Errors:                    678
   Search Artists:            1803604
   Known Summary IDs:         1455842


# Search For New Artists

In [ ]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [ ]:
useKnownData  = False
useMasterData = True

if useKnownData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].notna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useMasterData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].isna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  50696
#   Artist Names To Get:  42803
#   Artist Names To Get:  33587
#   Artist Names To Get:  21888
#   Artist Names To Get:  10630

## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "7:00pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue
    if searchedForErrors.get(artistName) is not None:
        continue

    try:
        response = webio.getArtistSearchData(artistName=artistName)
    except:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(60)
        continue
        
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(3.5)
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

## Save Results

In [ ]:
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        searchData = {}
        for searchTerm,searchTermData in mad.items():
            if isinstance(searchTermData,list):
                for item in searchTermData:
                    if isinstance(item,dict):
                        artistID = item['id'][2:] if isinstance(item.get('id'),str) else None
                        if artistID is not None:
                            searchData[artistID] = {k: v for k,v in item.items() if k in ['name','ref']}
        df = DataFrame(searchData).T
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [ ]:
mad = masterArtistsData.get()
df = getResponseDataFrame(mad)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = allmusic.MusicDBIO(local=False).data.getSearchArtistData()
    prevNewArtists = len(searchDF[~searchDF.index.isin(knownArtists.index)])
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF, df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("  ==> {0} Old Artists".format(searchDF[searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} New Artists".format(searchDF[~searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} Delta New Artists".format(len(searchDF[~searchDF.index.isin(knownArtists.index)])-prevNewArtists))
    print("Saving Data")
    allmusic.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)
    
#Found 1628828 Unique Results
#Found 1639245 Unique Results
#Found 1664572 Unique Results
#Found 1674908 Unique Results

In [ ]:
masterArtistsData.save(data={})

# Download Artist Data

In [6]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [15]:
useKnown = False

artistNames       = searchArtists
localArtistsDict  = localArtists.get()
artistNamesToGet  = artistNames[~artistNames.index.isin(localArtistsDict.keys())]
if useKnown is True:
    pdbio = PanDBIO()
    pdbio.setData()
    matchedIDs = pdbio.getNotNaDBIDs(db)[db]
    artistNamesToGet = artistNamesToGet[artistNamesToGet.index.isin(matchedIDs)]
    del pdbio

print("# {0} Search Results".format(db))
print("#   Available Names:      {0}".format(len(artistNames)))
print("#   Known Artist Names:   {0}".format(len(localArtistsDict)))
print("#   Artist Names To Get:  {0}".format(len(artistNamesToGet)))

#   Artist Names To Get:  223384

# AllMusic Search Results
#   Available Names:      1803604
#   Known Artist Names:   1590970
#   Artist Names To Get:  212634


In [16]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "9:50am")
#tt = TermTime("today", "10:00pm")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["name"]
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting AllMusic ArtistIDs] Start    ==> Time Is 2022-06-28 22:53:27
========================= termTime(day=tomorrow,time=9:50am) =========================
   ====> Terminate Time Set To 2022-06-29 09:50:00 <====
   ====> Will Terminate Process 10 Hours and 56 Minutes From Now
Getting Beech Street Band (0001307741)                    True
Getting The Blackburn-Beach Blues Band (0003330714)       True
Getting Les Bachi Bouzouk Band (0003788917)               True
Getting Chorale Saint Antoni Andante (0001875652)         True
Getting Saint Jake (0001217631)                           True
Getting David Rankin (0002151635)                         True
Getting Saint Jake (0002329614)                           True
Getting Jake Saint (0002054088)                           True
Getting Jake Saint John (0002725330)                      True
Getting Vitto Garcia (0000843539)                         True
Getting Vitto Cruz (0001490058)                           True
Getting Vitto (00008

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Major Seven Sound (0000405327)                    True
Getting Seven Ways To Sunday (0003234105)                 True
Getting Ark Noir (0003854963)                             True
Getting Doctor Saint Causey (0001970875)                  True
Getting Zine el Abadine (0001433621)                      True
Getting Arkimed (0000545054)                              True
Getting Arkymede (0002653678)                             True
Getting Arquimedes Arci (0000425620)                      True
Getting Argumedo Mato (0000497297)                        True
Getting Arquimedes Ara (0001041383)                       True
Getting Arquimedes Arcidiacono (0001065781)               True
Getting Arquimides Arcidiacono (0001339062)               True
Getting Arquimedes Arci (0001842768)                      True
Getting Arquimides Arce (0002405196)                      True
Getting Humberto Aargumedo (0002299638)                   True
Getting Lucas Argomedo (0003008954)                    

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Dafi Woo (0003964596)                             True
Getting Dafi (0002031005)                                 True
Getting Dafi (0004044980)                                 True
Getting Christina Saffran (0002215042)                    True
Getting Hillary Saffran (0004211185)                      True
Getting Christina Saffran Ashford (0000123835)            True
Getting Saffire (0004212879)                              True
Getting Sady Ramírez (0000828618)                         True
Getting Enrique Cid Ramirez (0001028711)                  True
Getting Sady Elias Ramirez Carrero (0003908438)           True
Getting Zion Lion (0002000692)                            True
Getting Zion Spirit (0000695975)                          True
Getting Zion Spirit (0001364036)                          True
Getting Spirit Zone (0002126314)                          True
Getting Arkhonia (0002517443)                             True
Getting Aregahegn Werashe (0002441436)                 

...

Getting Bill Crosbie (0003424562)                         True
Getting Henry Makobi (0000673750)                         True
Getting Der Manni (0003036413)                            True
Getting Manni der Libero (0002200280)                     True
Getting Orchester der Städtischen Bühnen Frankfurt am Main (0002193026)True
Getting Loose Head (0002061396)                           True
Getting Taylor Kennedy (0003147061)                       True
Getting Taylor Conti (0002534508)                         True
Getting Taylor Gundy (0003344524)                         True
Getting Taylor Kent (0003359635)                          True
Getting Taylor Kennedy (0003852929)                       True
Getting Kent Taylor (0001668497)                          True
Getting Candy Devine (0000653040)                         True
Getting Bill Clement (0001199816)                         True
Getting Bill Clements (0000492777)                        True
Getting Bill Clemmond (0002745216)        

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Confo (0003284930)                                True
Getting Dax Electron (0002971363)                         True
Getting Ariel Electron (0003097099)                       True
Getting Marco Gaggini (0003020562)                        True
Getting Pierre Gaggini (0002805296)                       True
Getting Piero Gaggini (0004196941)                        True
Getting Pat No (0003032828)                               True
Getting No Profile (0002768258)                           True
Getting Gaia & Luna (0001454364)                          True
Getting Der Process (0001754191)                          True
Getting Confraternite Delle Voci (0003422581)             True
Getting N Sisters (0001340809)                            True
Getting No Sister (0003865732)                            True
Getting Sister No Name (0002424324)                       True
Getting Pat Mabasa Na Shikange Sisters (0000136141)       True
Getting 250 (0003886537)                               

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

   ====> Terminate Time [2022-06-29 09:50:00] Is Reached <====
Process [Getting AllMusic ArtistIDs] Ran For 10 Hours and 56 Minutes    ==> Time Is 2022-06-29 09:50:24
Saving 1601770 AllMusic Artists Data
Saving 691 AllMusic Errors


In [ ]:
from lib.allmusic import moveLocalFiles
moveLocalFiles()

In [ ]:
tmp = []
for line in tmp:
    if line.startswith("Getting"):
        artistID = line.split()[-2][1:-1]
        print(artistID)
        del searchedForErrors[artistID]

# Create Tabs Data

In [ ]:
def getTabs(rData):
    extraData = rData.profile.extra
    tabs = extraData.get('tabs', []) if isinstance(extraData,dict) else []
    retval = {tab.title: tab.href for tab in tabs}
    return retval

In [ ]:
mio   = allmusic.MusicDBIO()
tabsData = None
for modVal in range(100):
    modValTabsData = mio.data.getModValData(modVal).apply(getTabs)
    tabsData = concat([tabsData, modValTabsData]) if isinstance(tabsData,Series) else modValTabsData

# Download Artist Discography Data

# Parse

In [ ]:
from utils import PoolIO
pio = PoolIO("AllMusic")
pio.parse(force=True)
pio.meta()
pio.sum()
pio.search()